# 01 - Exploration des Données : Jeu de Données Audio en Arabe Hassaniya

**Projet :** Système de Synthèse Vocale pour l'Arabe Hassaniya par Apprentissage par Transfert
**Auteur :** Mohamed Salem Ebnou Echvagha Oubeid (C34613)
**Module :** Dialectes NLP — Master M1 IA
**Date :** Juin 2026

---

## Objectif

Dans ce notebook, nous explorons notre jeu de données audio en arabe hassaniya pour comprendre :
- Combien d'échantillons nous avons
- La distribution des longueurs de texte
- Les propriétés audio (durée, taux d'échantillonnage, formes d'onde)
- La fréquence des caractères dans le texte hassaniya

Cette exploration est essentielle avant de construire tout système TTS — nous devons connaître
nos données pour prendre des décisions de prétraitement éclairées.

## 1. Configuration & Imports

In [ ]:
# Installer les dépendances (exécuter dans Colab)
# !pip install pandas pyarrow librosa soundfile matplotlib seaborn numpy tqdm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import struct
from collections import Counter

# Bibliothèques audio
try:
    import librosa
    import librosa.display
    import soundfile as sf
    AUDIO_AVAILABLE = True
    print('Bibliothèques audio chargées avec succès')
except ImportError:
    AUDIO_AVAILABLE = False
    print('Bibliothèques audio non disponibles — analyse textuelle uniquement')

try:
    from IPython.display import Audio, display
    IPYTHON_AVAILABLE = True
except ImportError:
    IPYTHON_AVAILABLE = False

# Paramètres des graphiques
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

print(f'Configuration terminée')

## 2. Chargement du Jeu de Données

Notre jeu de données est stocké sous forme de **fichier Parquet** — un format colonnaire qui stocke
efficacement à la fois du texte et des données binaires (octets audio). Chaque ligne contient :
- `audio` : un dictionnaire avec `bytes` (données audio brutes) et `path`
- `transcription` : le texte en arabe hassaniya

In [ ]:
# Charger le jeu de données
df = pd.read_parquet('../audios_dataset.parquet')

print(f'Forme du jeu de données : {df.shape}')
print(f'Colonnes : {list(df.columns)}')
print(f'\nTypes des colonnes :')
print(df.dtypes)
print(f'\nNombre total d\'échantillons : {len(df)}')

In [ ]:
# Examiner la structure de la colonne audio
sample = df['audio'].iloc[0]
print(f'Type de la colonne audio : {type(sample)}')
if isinstance(sample, dict):
    print(f'Clés du dictionnaire audio : {list(sample.keys())}')
    audio_bytes = sample.get('bytes', b'')
    print(f'Taille des octets audio : {len(audio_bytes)} octets')
    print(f'Chemin audio : {sample.get("path", "Aucun")}')

## 3. Analyse Textuelle

Analysons les transcriptions pour comprendre les caractéristiques linguistiques
de notre jeu de données hassaniya.

In [ ]:
# Afficher des exemples de transcriptions
print('=== Exemples de Transcriptions Hassaniya ===')
print()
for i in range(min(10, len(df))):
    print(f'  [{i:3d}] {df["transcription"].iloc[i]}')

In [ ]:
# Statistiques de longueur de texte
df['text_length'] = df['transcription'].str.len()
df['word_count'] = df['transcription'].str.split().str.len()

print('=== Statistiques de Longueur de Texte ===')
print(f'Longueur moyenne (caractères) : {df["text_length"].mean():.1f}')
print(f'Longueur médiane (caractères) : {df["text_length"].median():.1f}')
print(f'Longueur minimale : {df["text_length"].min()}')
print(f'Longueur maximale : {df["text_length"].max()}')
print(f'Écart-type : {df["text_length"].std():.1f}')
print()
print('=== Statistiques du Nombre de Mots ===')
print(f'Moyenne de mots par échantillon : {df["word_count"].mean():.1f}')
print(f'Maximum de mots par échantillon : {df["word_count"].max()}')
print(f'Minimum de mots par échantillon : {df["word_count"].min()}')

In [ ]:
# Visualiser la distribution des longueurs de texte
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution de la longueur en caractères
axes[0].hist(df['text_length'], bins=30, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Longueur (caractères)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des Longueurs de Transcriptions (Caractères)')
axes[0].axvline(df['text_length'].mean(), color='red', linestyle='--', label=f'Moyenne : {df["text_length"].mean():.0f}')
axes[0].legend()

# Distribution du nombre de mots
axes[1].hist(df['word_count'], bins=20, color='#4CAF50', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Nombre de Mots')
axes[1].set_ylabel('Fréquence')
axes[1].set_title('Distribution du Nombre de Mots par Échantillon')
axes[1].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f'Moyenne : {df["word_count"].mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée dans results/figures/text_length_distribution.png')

In [ ]:
# Analyse de fréquence des caractères
all_text = ' '.join(df['transcription'].tolist())
char_counts = Counter(ch for ch in all_text if ch.strip())

# Top 30 des caractères les plus fréquents
top_chars = char_counts.most_common(30)
chars, counts = zip(*top_chars)

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(range(len(chars)), counts, color='#FF9800', edgecolor='white')
ax.set_xticks(range(len(chars)))
ax.set_xticklabels(chars, fontsize=14)
ax.set_xlabel('Caractère')
ax.set_ylabel('Fréquence')
ax.set_title('Top 30 des Caractères les Plus Fréquents dans les Transcriptions Hassaniya')

plt.tight_layout()
plt.savefig('../results/figures/character_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nNombre total de caractères uniques : {len(char_counts)}')
print(f'Nombre total de caractères (hors espaces) : {sum(char_counts.values())}')

## 4. Analyse Audio

Examinons maintenant les propriétés audio de notre jeu de données.
Chaque échantillon audio est stocké sous forme d'octets bruts — nous les décodons pour analyser
les formes d'onde, les durées et les propriétés spectrales.

In [ ]:
def decode_audio(audio_dict, target_sr=22050):
    """Décoder les octets audio du jeu de données en tableau de forme d'onde."""
    audio_bytes = audio_dict.get('bytes', b'')
    if not audio_bytes or not AUDIO_AVAILABLE:
        return None, None
    try:
        audio, sr = sf.read(io.BytesIO(audio_bytes))
        if sr != target_sr:
            audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=target_sr)
            sr = target_sr
        return audio.astype(np.float32), sr
    except Exception as e:
        print(f'Erreur de décodage audio : {e}')
        return None, None

# Décoder le premier échantillon comme test
if AUDIO_AVAILABLE:
    test_audio, test_sr = decode_audio(df['audio'].iloc[0])
    if test_audio is not None:
        print(f'Audio décodé avec succès')
        print(f'Taux d\'échantillonnage : {test_sr} Hz')
        print(f'Durée : {len(test_audio)/test_sr:.2f} secondes')
        print(f'Forme de la forme d\'onde : {test_audio.shape}')
    else:
        print('Impossible de décoder l\'audio — essai d\'une méthode alternative')
else:
    print('Bibliothèques audio non disponibles')

In [ ]:
# Calculer les durées pour tous les échantillons
if AUDIO_AVAILABLE:
    durations = []
    sample_rates = []
    audio_sizes = []
    
    for i in range(len(df)):
        audio_dict = df['audio'].iloc[i]
        audio_bytes = audio_dict.get('bytes', b'')
        audio_sizes.append(len(audio_bytes))
        
        try:
            audio, sr = sf.read(io.BytesIO(audio_bytes))
            durations.append(len(audio) / sr)
            sample_rates.append(sr)
        except:
            durations.append(0)
            sample_rates.append(0)
    
    df['duration'] = durations
    df['sample_rate'] = sample_rates
    df['audio_size_kb'] = [s / 1024 for s in audio_sizes]
    
    valid = df[df['duration'] > 0]
    print('=== Statistiques de Durée Audio ===')
    print(f'Échantillons audio valides : {len(valid)} / {len(df)}')
    print(f'Durée moyenne : {valid["duration"].mean():.2f} secondes')
    print(f'Durée minimale : {valid["duration"].min():.2f} secondes')
    print(f'Durée maximale : {valid["duration"].max():.2f} secondes')
    print(f'Durée totale : {valid["duration"].sum():.1f} secondes ({valid["duration"].sum()/60:.1f} minutes)')
    print(f'Taux d\'échantillonnage trouvés : {sorted(valid["sample_rate"].unique())}')
else:
    print('Analyse audio ignorée — librosa/soundfile non disponibles')

In [ ]:
# Visualiser les durées audio
if AUDIO_AVAILABLE and 'duration' in df.columns:
    valid = df[df['duration'] > 0]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogramme des durées
    axes[0].hist(valid['duration'], bins=30, color='#9C27B0', edgecolor='white', alpha=0.8)
    axes[0].set_xlabel('Durée (secondes)')
    axes[0].set_ylabel('Fréquence')
    axes[0].set_title('Distribution des Durées Audio')
    axes[0].axvline(valid['duration'].mean(), color='red', linestyle='--', 
                    label=f'Moyenne : {valid["duration"].mean():.1f}s')
    axes[0].legend()
    
    # Nuage de points longueur de texte vs durée
    axes[1].scatter(valid['text_length'], valid['duration'], alpha=0.5, color='#009688', s=30)
    axes[1].set_xlabel('Longueur du Texte (caractères)')
    axes[1].set_ylabel('Durée Audio (secondes)')
    axes[1].set_title('Longueur du Texte vs Durée Audio')
    
    plt.tight_layout()
    plt.savefig('../results/figures/audio_duration_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure sauvegardée dans results/figures/audio_duration_analysis.png')

In [ ]:
# Visualiser des formes d'onde et spectrogrammes
if AUDIO_AVAILABLE:
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    
    for idx in range(3):
        audio, sr = decode_audio(df['audio'].iloc[idx])
        if audio is None:
            continue
        
        # Forme d'onde
        axes[idx, 0].plot(np.linspace(0, len(audio)/sr, len(audio)), audio, 
                          color='#1565C0', linewidth=0.5)
        axes[idx, 0].set_title(f'Forme d\'onde — Échantillon {idx}', fontsize=10)
        axes[idx, 0].set_xlabel('Temps (s)')
        axes[idx, 0].set_ylabel('Amplitude')
        
        # Mel spectrogramme
        mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=80, n_fft=1024, hop_length=256)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        img = librosa.display.specshow(mel_db, y_axis='mel', x_axis='time', 
                                       sr=sr, hop_length=256, ax=axes[idx, 1])
        axes[idx, 1].set_title(f'Mel Spectrogramme — Échantillon {idx}', fontsize=10)
    
    plt.tight_layout()
    plt.savefig('../results/figures/sample_waveforms_spectrograms.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure sauvegardée dans results/figures/sample_waveforms_spectrograms.png')

In [ ]:
# Écouter des échantillons (fonctionne dans Jupyter/Colab)
if AUDIO_AVAILABLE and IPYTHON_AVAILABLE:
    for i in range(min(3, len(df))):
        audio, sr = decode_audio(df['audio'].iloc[i])
        if audio is not None:
            print(f'\nÉchantillon {i} : "{df["transcription"].iloc[i]}"')
            display(Audio(audio, rate=sr))

## 5. Résumé du Jeu de Données

Compilons un résumé complet de notre jeu de données.

In [ ]:
print('=' * 60)
print('     JEU DE DONNÉES AUDIO HASSANIYA — RÉSUMÉ')
print('=' * 60)
print(f'  Nombre total d\'échantillons : {len(df)}')
print(f'  Transcriptions uniques :      {df["transcription"].nunique()}')
print(f'  Caractères uniques :          {len(char_counts)}')
print(f'  Longueur moy. du texte :      {df["text_length"].mean():.1f} car.')
print(f'  Nombre moy. de mots :         {df["word_count"].mean():.1f} mots')
if 'duration' in df.columns:
    valid = df[df['duration'] > 0]
    print(f'  Échantillons audio valides :  {len(valid)}')
    print(f'  Durée audio totale :          {valid["duration"].sum():.1f}s ({valid["duration"].sum()/60:.1f} min)')
    print(f'  Durée audio moyenne :         {valid["duration"].mean():.2f}s')
print('=' * 60)
print()
print('Observations clés :')
print('  - Jeu de données petit mais utilisable pour un MVP/preuve de concept')
print('  - Le texte est en dialecte hassaniya informel (pas de l\'ASM)')
print('  - Énoncés de longueur variable — bonne diversité')
print('  - L\'apprentissage par transfert est nécessaire vu la taille du jeu de données')

## Conclusion

Notre jeu de données hassaniya contient **294 paires audio-texte** — une collection
petite mais significative pour un système TTS de preuve de concept. Les données montrent :

1. **Diversité textuelle** : Les transcriptions vont de 4 à 182 caractères
2. **Caractéristiques dialectales** : Arabe hassaniya informel, distinct de l'Arabe Standard Moderne
3. **Qualité audio** : Durées variables, adaptées au développement d'un pipeline de prétraitement

**Prochaine étape** : Notebook 02 — Annotation et normalisation du texte.